In [1]:
!pip install -q datasets sentence-transformers

import numpy as np, pandas as pd, torch
from datasets import load_dataset
from sentence_transformers import SentenceTransformer

device = "cuda" if torch.cuda.is_available() else "cpu"

# 1. Load 5,000 reviews
ds = load_dataset("Yelp/yelp_review_full")
df = pd.DataFrame(ds["test"].shuffle(seed=42).select(range(5000)))
df["stars"] = df["label"] + 1

# 2. Turn every review into a vector that captures its meaning
st = SentenceTransformer("all-MiniLM-L6-v2", device=device)
emb = st.encode(df["text"].tolist(), batch_size=128,
                show_progress_bar=True, normalize_embeddings=True)

# 3. Search: embed the query and find the closest reviews
def search(query, k=5, max_stars=None):
    q = st.encode([query], normalize_embeddings=True)
    scores = (emb @ q.T).ravel()
    if max_stars is not None:
        scores = np.where(df.stars.values <= max_stars, scores, -1)
    print(f'QUERY: "{query}"' + (f"  (only {max_stars} stars or fewer)" if max_stars else ""))
    for j in scores.argsort()[::-1][:k]:
        print(f"  [{df.stars.iloc[j]}★ | score {scores[j]:.2f}] "
              f"{df.text.iloc[j][:200]}".replace("\n", " "))
    print()

search("the waiter ignored us and the food arrived cold")
search("great place for a quiet date night")
search("rude staff and billing problems", max_stars=2)
search("friendly and fast service", k=5)


README.md:   0%|          | 0.00/6.72k [00:00<?, ?B/s]

yelp_review_full/train-00000-of-00001.pa(…): reconstructing file:   0%|          |  0.00B /  299MB            

yelp_review_full/train-00000-of-00001.pa(…): downloading bytes:           |  0.00B            

yelp_review_full/test-00000-of-00001.par(…): reconstructing file:   0%|          |  0.00B / 23.5MB            

yelp_review_full/test-00000-of-00001.par(…): downloading bytes:           |  0.00B            

...

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Batches:   0%|          | 0/40 [00:00<?, ?it/s]

QUERY: "the waiter ignored us and the food arrived cold"
  [1★ | score 0.61] Food took forever to cone out and was cold when it did. Nobody ever checked on us and we saw our waitress a whole three times between our food arriving and paying our bill.  Despite the restaurant bei
  [2★ | score 0.52] The service of our waitress was very rude and not professional.  Sat for 20min never came back to take our order. Food didn't taste good. It was too salty.
  [1★ | score 0.50] Food is OK after you finally get it. Waited 45 min for breakfast and my girlfriend's food was cold. They said no problem, it will only take a minute to replace....took 10. Friendly but incapable servi
  [2★ | score 0.49] The hostess didn't ask us where we wanted to sit and forgot to set our table. Our waitress was very good but seemed to be too busy. They need better management to train their people to work as a team.
  [1★ | score 0.49] Arrived @ 6:15pm on Sat without a reservation, restaurant was fairly full.  We were 